In [1]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path= "sms_spam_collection.zip"
extracted_path= "sms_spam_collection"
data_file_path= Path(extracted_path)  / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(
        url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():  
        print(f"{data_file_path} already exists. Skipping download"
              "and extraction."
              )
        return

    
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb" ) as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path= Path(extracted_path)/ "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"file donwloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)




sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping downloadand extraction.


In [2]:
import pandas as pd
df= pd.read_csv(
    data_file_path, sep= "\t", header= None, names= ["Label", "Text"]
)
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [3]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [4]:

print( df[df["Label"]== "spam"]  )

     Label                                               Text
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
5     spam  FreeMsg Hey there darling it's been 3 week's n...
8     spam  WINNER!! As a valued network customer you have...
9     spam  Had your mobile 11 months or more? U R entitle...
11    spam  SIX chances to win CASH! From 100 to 20,000 po...
...    ...                                                ...
5537  spam  Want explicit SEX in 30 secs? Ring 02073162414...
5540  spam  ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547  spam  Had your contract mobile 11 Mnths? Latest Moto...
5566  spam  REMINDER FROM O2: To get 2.50 pounds free call...
5567  spam  This is the 2nd time we have tried 2 contact u...

[747 rows x 2 columns]


In [5]:
def create_balanced_dataset(df):
    num_spam= df[df["Label"]== "spam"].shape[0]
    ham_subset= df[df["Label"]== "ham"].sample(
        num_spam, random_state=123
    )

    balanced_df = pd.concat([
        ham_subset, df[df["Label"]== "spam"]
    ])
    return balanced_df

balanced_df= create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [6]:
balanced_df["Label"]= balanced_df["Label"].map({"ham":0, "spam":1})

In [7]:
def random_split(df, train_frac, validation_frac):

    df= df.sample(
        frac=1, random_state= 123
    ).reset_index(drop = True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[: train_end]
    validation_df= df[train_end: validation_end]

    test_df= df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df= random_split(
    balanced_df, 0.7, 0.1
)

In [8]:
train_df.to_csv("train.csv", index= None)
validation_df.to_csv("validation_csv", index= None)
test_df.to_csv("test_csv", index= None)

In [9]:
import tiktoken
tokenizer= tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special= {"<|endoftext|>"}))

[50256]


In [10]:
print(tokenizer.decode([50256]))

<|endoftext|>


In [11]:
print(tokenizer.decode([91, 437, 1659, 5239, 91]))

|endoftext|


dataset class

In [25]:
import torch
import pandas as pd
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # 1. Encode all texts
        self.encoded_texts = [
            tokenizer.encode(text)
            for text in self.data["Text"]
        ]

        # Determine maximum sequence length
        if max_length is None:
            self.max_length = self.longest_encoded_length()
        else:
            self.max_length = max_length

        # 2. Truncate sequences
        self.encoded_texts = [
            encoded_text[:self.max_length]
            for encoded_text in self.encoded_texts
        ]

        # 3. Pad sequences
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long),
        )

    def __len__(self):
        return len(self.data)

    def longest_encoded_length(self):
        max_length = 0

        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length

        return max_length

In [17]:
data= pd.read_csv("train.csv")
data

,Label,Text
0,0,Dude how do you like the buff wind.
1,0,Tessy..pls do me a favor. Pls convey my birthd...
2,1,Reminder: You have not downloaded the content ...
3,1,Got what it takes 2 take part in the WRC Rally...
4,1,"Shop till u Drop, IS IT YOU, either 10K, 5K, £..."
...,...,...
1040,1,4mths half price Orange line rental & latest c...
1041,1,Thanks for the Vote. Now sing along with the s...
1042,1,IMPORTANT INFORMATION 4 ORANGE USER 0796XXXXXX...
1043,1,Urgent! call 09066612661 from landline. Your c...


In [24]:
for i, text in enumerate(data["Text"]):
    print(i, text , "\n")

    if(i>10):
        break
    

0 Dude how do you like the buff wind. 

1 Tessy..pls do me a favor. Pls convey my birthday wishes to Nimya..pls dnt forget it. Today is her birthday Shijas 

2 Reminder: You have not downloaded the content you have already paid for. Goto http://doit. mymoby. tv/ to collect your content. 

3 Got what it takes 2 take part in the WRC Rally in Oz? U can with Lucozade Energy! Text RALLY LE to 61200 (25p), see packs or lucozade.co.uk/wrc & itcould be u! 

4 Shop till u Drop, IS IT YOU, either 10K, 5K, £500 Cash or £100 Travel voucher, Call now, 09064011000. NTT PO Box CR01327BT fixedline Cost 150ppm mobile vary 

5 Am on a train back from northampton so i'm afraid not! I'm staying skyving off today ho ho! Will be around wednesday though. Do you fancy the comedy club this week by the way? 

6 Dude. What's up. How Teresa. Hope you have been okay. When i didnt hear from these people, i called them and they had received the package since dec  &lt;#&gt; . Just thot you'ld like to know. Do have a 

In [26]:
train_dataset= SpamDataset(
    csv_file= "train.csv",
    max_length= None,
    tokenizer= tokenizer
)

In [ ]:
print(train_dataset.)